%md
### AutoML

In [ ]:
%pip install "flaml[automl]" --quiet

In [ ]:
dbutils.library.restartPython()

In [ ]:
import mlflow
from flaml import AutoML
import pandas as pd
from sklearn.utils.class_weight import compute_sample_weight

# 1. Load Data
train_df = spark.read.table("crypto_db.gold_late_capital_72h")
train_pdf = train_df.toPandas()

# 2. Sort by time (the most important first step for time series models)
train_pdf = train_pdf.sort_values('open_time_ts').reset_index(drop=True)

# 🚨 KEY FIX 1: Feature Whitelist
FEATURES = [
    "atr_pct", 
    "volatility_4h", 
    "bias_to_ma_4h", 
    "days_since_signal", 
    "atr_24h_pct", 
    "volume_surge_ratio", 
    "lower_wick_ratio", 
    "roc_4h", 
    "trend_extension_pct"
]

X_all = train_pdf[FEATURES]
y_all = train_pdf['best_strategy_72h']

print(f"[INFO] Number of features: {len(FEATURES)}. Removed 'close' price to prevent extrapolation trap.")

# ==========================================================
# 🚨 KEY FIX 2: Manual Time Split to bypass FLAML's Bug
# ==========================================================
# Manually split into 90% (Train) and 10% (Validation)
split_idx = int(len(train_pdf) * 0.9)

X_tr = X_all.iloc[:split_idx]
y_tr = y_all.iloc[:split_idx]

X_va = X_all.iloc[split_idx:]
y_va = y_all.iloc[split_idx:]

# 👑 Professional detail: weights should "only" be calculated from training set (y_tr) to avoid Data Leakage
sample_weights_tr = compute_sample_weight(class_weight='balanced', y=y_tr)

print(f"[INFO] Manual split completed -> Training set: {len(X_tr)} records | Validation set: {len(X_va)} records")

# 3. Configure FLAML
automl = AutoML()
automl_settings = {
    "time_budget": 300,
    "metric": 'macro_f1',
    "task": 'classification',
    "log_file_name": "late_capital_automl.log"
    # 🚨 Note: We removed eval_method and split_type because we will directly feed X_va to it
}

# 4. Train Model with MLflow tracking
# Define model name
model_name = "crypto_execution_optimizer"
mlflow.set_experiment("/Shared/late_capital_experiment") 

with mlflow.start_run(run_name="FLAML_Manual_Split_Balanced"):
    
    # Feed the split data, validation set, and weights precisely to FLAML
    automl.fit(
        X_train=X_tr, 
        y_train=y_tr, 
        sample_weight=sample_weights_tr, 
        X_val=X_va,    # <--- Explicitly specify validation set, FLAML won't split randomly anymore
        y_val=y_va, 
        **automl_settings
    )
    
    print("\n[SUCCESS] FLAML AutoML training completed!")
    print(f"👑 Best model algorithm: {automl.best_estimator}")
    print(f"📈 Best validation F1 Score: {1 - automl.best_loss:.4f}")
    
    # Log best parameters and scores
    mlflow.log_params(automl.best_config)
    mlflow.log_metric("val_macro_f1", 1 - automl.best_loss)
    
    # Generate Signature to ensure Schema validation passes
    from mlflow.models.signature import infer_signature
    sample_input = X_tr.head(5)
    signature = infer_signature(sample_input, automl.predict(sample_input))
    
    # Extract sklearn model and register
    best_sklearn_model = automl.model.estimator
    mlflow.sklearn.log_model(
        sk_model=best_sklearn_model,
        artifact_path="model",
        signature=signature,
        registered_model_name=model_name
    )

print(f"\n[SUCCESS] Model has successfully avoided the Bug, trained and registered!")

In [ ]:
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature

# 1. Define the name of the model to be registered
model_name = "crypto_execution_optimizer"

# 2. Set the MLflow experiment path
mlflow.set_experiment("/Shared/late_capital_experiment") 

# Start a new Run to log and register the corrected model
with mlflow.start_run(run_name="FLAML_ExtraTree_Run_Fixed_Features") as run:
    
    # Log the best hyperparameters and validation score (Great job on this step!)
    mlflow.log_params(automl.best_config)
    mlflow.log_metric("val_macro_f1", 1 - automl.best_loss)
    
    # 👑 Key Step: Infer Model Signature. 
    # This ensures the Serving Endpoint knows it now only accepts 9 features.
    sample_input = X_train.head(5)
    sample_output = automl.predict(sample_input)
    signature = infer_signature(sample_input, sample_output)
    
    # Extract the pure scikit-learn model from the FLAML wrapper to optimize API speed
    best_sklearn_model = automl.model.estimator
    
    # Log the model to MLflow and automatically "Register" it to generate a new version
    model_info = mlflow.sklearn.log_model(
        sk_model=best_sklearn_model,
        artifact_path="model",
        signature=signature,
        registered_model_name=model_name
    )

print(f"\n[SUCCESS] Model has been successfully logged and registered!")
print(f"MLflow Run ID: {run.info.run_id}")
print(f"Please go to the 'Models' tab on the left and look for '{model_name}' (you should see a brand new version number).")